In [52]:
# make sure jupyter server is installed in the environment
# then install dependencies
%pip install pandas nltk scikit-learn numpy matplotlib --quiet

# make sure all dependencies are installed
import pandas as pd
import nltk
from nltk import word_tokenize
import sklearn as sk
import numpy as np
import matplotlib.pyplot as plt
from nltk.stem import PorterStemmer, WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer

# download nltk resources
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

# loading the datasets
negative_df = pd.read_csv('./data/processedNegative.csv', header=None)
positive_df = pd.read_csv('./data/processedPositive.csv', header=None)
neutral_df = pd.read_csv('./data/processedNeutral.csv', header=None)

# transposing the datasets to have tweets in rows
negative_df = negative_df.transpose().rename(columns={0: 'tweet'})
positive_df = positive_df.transpose().rename(columns={0: 'tweet'})
neutral_df = neutral_df.transpose().rename(columns={0: 'tweet'})

# merge all datasets into a single dataframe with a sentiment label
neutral_df['sentiment'] = 'neutral'
positive_df['sentiment'] = 'positive'
negative_df['sentiment'] = 'negative'
all_tweets_df = pd.concat([neutral_df, positive_df, negative_df]).reset_index(drop=True)

# drop any rows with missing values
all_tweets_df = all_tweets_df.dropna()

# display the first few rows with random sentiment tweets
all_tweets_df.sample(10).reset_index(drop=True)

Note: you may need to restart the kernel to use updated packages.


[nltk_data] Downloading package punkt_tab to /home/samy/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /home/samy/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /home/samy/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


,tweet,sentiment
0,Free market is tops!!! Real human rights sound...,positive
1,Give away Random RT! for a chance to win a 10U...,positive
2,Did you notice? There is no killings of RSS / ...,positive
3,You mean your voice changed? unhappy,negative
4,Not surprised that I have the least amount of ...,negative
5,last night I had a dream in which I moved to ...,negative
6,Yogendra Yadav on the future of India under wh...,neutral
7,What Supreme Court said about welfare schemes ...,neutral
8,it looks like iOS 11 is due to kill it unhappy,negative
9,Imagine paying a of Rs500 -- in 1863.,neutral


### Text Preprocessing

we're going to generate multiple datasets using different preprocessing techniques

##### 1. Text Cleaning

In [53]:
def remove_mentions(tweet):
    """Remove Twitter mentions from a tweet."""
    return ' '.join(word for word in tweet.split() if not word.startswith('@'))

def remove_urls(tweet):
    """Remove URLs from a tweet."""
    return ' '.join(word for word in tweet.split() if not word.startswith('http'))

def remove_hashtags(tweet):
    """Remove hashtags from a tweet."""
    return ' '.join(word for word in tweet.split() if not word.startswith('#'))

def remove_punctuation(tweet):
    """Remove punctuation from a tweet."""
    import string
    return tweet.translate(str.maketrans('', '', string.punctuation))

def convert_to_lowercase(tweet):
    """Convert all characters in a tweet to lowercase."""
    return tweet.lower()

# create a cleaner dataset
def clean_tweet(tweet):
    tweet = str(tweet)
    tweet = remove_mentions(tweet)
    tweet = remove_urls(tweet)
    tweet = remove_hashtags(tweet)
    tweet = remove_punctuation(tweet)
    tweet = convert_to_lowercase(tweet)
    return tweet

In [54]:
# apply cleaning
all_tweets_df['tweet'] = all_tweets_df['tweet'].apply(clean_tweet)

# drop null values that may have been introduced during cleaning
all_tweets_df_cleaned = all_tweets_df.dropna().reset_index(drop=True)

display(all_tweets_df_cleaned.sample(10).reset_index(drop=True))
# display count of tweets in each cleaned dataset
print(f"Neutral tweets: {len(all_tweets_df_cleaned[all_tweets_df_cleaned['sentiment'] == 'neutral'])}")
print(f"Positive tweets: {len(all_tweets_df_cleaned[all_tweets_df_cleaned['sentiment'] == 'positive'])}")
print(f"Negative tweets: {len(all_tweets_df_cleaned[all_tweets_df_cleaned['sentiment'] == 'negative'])}")

,tweet,sentiment
0,a huge thank you to all the within the social ...,positive
1,its only been a few days since their promotion...,negative
2,supreme court shoots down govt bid to put spor...,neutral
3,aap 6,neutral
4,says sp leader shivpal yadav,neutral
5,why are you whimper at me,negative
6,koalas are dying of thirst and its all because...,negative
7,so upsetting the abuse these kids get unhappy 3,negative
8,seems you like my black lingerie picture then ...,positive
9,govt drops plan to slip in pseudo,neutral


Neutral tweets: 1569
Positive tweets: 1183
Negative tweets: 1116


#### 2. Tokenizer

###### 2.1 Basic Tokenization

In [55]:
def tokenize_tweet(tweet):
    """Tokenize a tweet into words."""
    return word_tokenize(tweet)

###### 2.2 Stemming

In [56]:
stemmer = PorterStemmer()

def stem_tokens(tokens):
    """Stem a list of tokens."""
    return [stemmer.stem(token) for token in tokens]

###### 2.3 Lemmatization

In [57]:
lemmatizer = WordNetLemmatizer()

def lemmatize_tokens(tokens):
    """Lemmatize a list of tokens."""
    return [lemmatizer.lemmatize(token) for token in tokens]

##### Split train/test data

In [58]:
# split datasets into training and testing sets (80% train, 20% test)



### Vectorization

##### 1. Bag of Words

In [59]:

def custom_tokenizer(text, ):
    """Custom tokenizer that returns pre-tokenized input."""
    # using stemming by default
    return stem_tokens(tokenize_tweet(text))

def count_vectorize(tweets):
    """Create a Bag of Words representation of the cleaned tweets in the dataframe."""
    vectorizer = CountVectorizer(tokenizer=custom_tokenizer, lowercase=False, stop_words='english')
    bow_matrix = vectorizer.fit_transform(tweets)
    return vectorizer, bow_matrix

vectorizer, bow_matrix = count_vectorize(all_tweets_df_cleaned['tweet'])

x_train, x_test, y_train, y_test = train_test_split(
    bow_matrix,
    all_tweets_df_cleaned['sentiment'],
    test_size=0.2,
    random_state=42
)

/home/samy/tweets/.venv/lib/python3.12/site-packages/sklearn/feature_extraction/text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/home/samy/tweets/.venv/lib/python3.12/site-packages/sklearn/feature_extraction/text.py:411: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['abov', 'afterward', 'alon', 'alreadi', 'alway', 'ani', 'anoth', 'anyon', 'anyth', 'anywher', 'becam', 'becaus', 'becom', 'befor', 'besid', 'cri', 'describ', 'dure', 'els', 'elsewher', 'empti', 'everi', 'everyon', 'everyth', 'everywher', 'fifti', 'formerli', 'forti', 'ha', 'henc', 'hereaft', 'herebi', 'hi', 'howev', 'hundr', 'inde', 'latterli', 'mani', 'meanwhil', 'moreov', 'mostli', 'nobodi', 'noon', 'noth', 'nowher', 'onc', 'onli', 'otherwis', 'ourselv', 'perhap', 'pleas', 'seriou', 'sever', 'sinc', 'sincer', 'sixti', 'someon', 'someth', 'sometim', 'somewher', 'themselv', 'thenc'

### Cosine Similarity

### Machine Learning

In [60]:
# start with Random Forest Classifier
from sklearn.ensemble import RandomForestClassifier
rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42)
rf_classifier.fit(x_train, y_train)
# evaluate on test set
test_predictions = rf_classifier.predict(x_test)
from sklearn.metrics import classification_report
print("Classification Report:")
print(classification_report(y_test, test_predictions))

Classification Report:
              precision    recall  f1-score   support

    negative       0.89      0.84      0.86       221
     neutral       0.86      0.94      0.90       321
    positive       0.91      0.85      0.88       232

    accuracy                           0.88       774
   macro avg       0.89      0.88      0.88       774
weighted avg       0.89      0.88      0.88       774



In [61]:
# test a few sample tweets
sample_tweets = [
    "I love this product! It's amazing.",
    "This is the worst service I've ever experienced.",
    "It's okay, not great but not terrible either.",
    "Absolutely fantastic! Exceeded my expectations.",
    "I'm really disappointed with the quality.",
]

sample_bow = vectorizer.transform(sample_tweets)
sample_predictions = rf_classifier.predict(sample_bow)
for tweet, sentiment in zip(sample_tweets, sample_predictions):
    print(f"Tweet: {tweet}\nPredicted Sentiment: {sentiment}\n")

Tweet: I love this product! It's amazing.
Predicted Sentiment: positive

Tweet: This is the worst service I've ever experienced.
Predicted Sentiment: negative

Tweet: It's okay, not great but not terrible either.
Predicted Sentiment: positive

Tweet: Absolutely fantastic! Exceeded my expectations.
Predicted Sentiment: neutral

Tweet: I'm really disappointed with the quality.
Predicted Sentiment: positive

